# NSDMB Landslide Data Check

This notebook reads the NSDMB geodatabase from `common_incoming_data` and checks whether it contains landslide-related data.


In [ ]:
from pathlib import Path
import re
import pandas as pd
import geopandas as gpd
import fiona
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)


In [ ]:
# Paths
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
nsdmb_gdb = base_path / 'dphil_common_cross_cutting/common_incoming_data/nsdmb/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb'

print('NSDMB geodatabase exists:', nsdmb_gdb.exists())
print('Path:', nsdmb_gdb)


In [ ]:
# List all layers and find landslide-related ones
all_layers = fiona.listlayers(nsdmb_gdb)
print(f'Total layers in geodatabase: {len(all_layers)}')

keywords = ['landslide', 'slope', 'mass movement', 'debris', 'rockfall', 'slip']
pattern = re.compile('|'.join(re.escape(k) for k in keywords), flags=re.IGNORECASE)

landslide_layers = [layer for layer in all_layers if pattern.search(layer)]
print(f'Landslide-related layer names found: {len(landslide_layers)}')
for layer in landslide_layers:
    print(' -', layer)


In [ ]:
# Summarize landslide-related layers (feature count, geometry type, first fields)
summary_rows = []
for layer in landslide_layers:
    with fiona.open(nsdmb_gdb, layer=layer) as src:
        props = list(src.schema.get('properties', {}).keys())
        summary_rows.append({
            'layer': layer,
            'feature_count': len(src),
            'geometry_type': src.schema.get('geometry'),
            'sample_fields': ', '.join(props[:12])
        })

layer_summary_df = pd.DataFrame(summary_rows).sort_values(['feature_count', 'layer'], ascending=[False, True])
layer_summary_df


In [ ]:
# Quick preview of key layers
preview_layers = [
    'landslide_historical_events_1895to2009',
    'HazVul_LandslideInventory_KSA',
    'HazVul_LandslideInventory_StThomas',
    'landslide_vulnerability'
]

for layer in preview_layers:
    if layer in landslide_layers:
        gdf = gpd.read_file(nsdmb_gdb, layer=layer)
        print(f"
Layer: {layer} | rows: {len(gdf)}")
        display(gdf.head(3))


In [ ]:
# Basic checks on historical landslide events
hist_layer = 'landslide_historical_events_1895to2009'
if hist_layer in landslide_layers:
    hist = gpd.read_file(nsdmb_gdb, layer=hist_layer)
    print('Rows:', len(hist))
    if 'YEAR' in hist.columns:
        print('Year min/max:', hist['YEAR'].min(), hist['YEAR'].max())
    if 'PARISH' in hist.columns:
        display(hist['PARISH'].value_counts().head(15))


In [ ]:
# Simple map of selected landslide layers
map_layers = [
    'HazVul_LandslideInventory_KSA',
    'HazVul_LandslideInventory_StMary',
    'HazVul_LandslideInventory_StThomas',
    'landslide_historical_events_1895to2009'
]

fig, ax = plt.subplots(figsize=(8, 8))
for layer in map_layers:
    if layer in landslide_layers:
        gdf = gpd.read_file(nsdmb_gdb, layer=layer)
        gdf.plot(ax=ax, markersize=2, alpha=0.5, label=layer)

ax.set_title('NSDMB Landslide-related Layers (selected)')
ax.set_axis_off()
ax.legend(loc='lower left', fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# Final check message
if len(landslide_layers) > 0:
    print('Yes: NSDMB includes landslide-related data.')
else:
    print('No landslide-related layer names found in NSDMB geodatabase.')
